# 02 — HMM Decoding and State-Specific FC

Decode HMM states, split time series by subject, and compute per-subject/state functional connectivity matrices (soft gamma weighting by default).

In [1]:
import json
import pickle
from pathlib import Path
from typing import Dict

import numpy as np
import pandas as pd
import yaml
import os

PROJECT_ROOT = Path('..').resolve()
CONFIG_NAME = os.environ.get('GRAPH_THEORY_CONFIG', 'config.yaml')
with open(PROJECT_ROOT / CONFIG_NAME, encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

use_example_data = bool(cfg.get('use_example_data', False))
paths = cfg['paths']
analysis = cfg['analysis']
DATA_DIR = (PROJECT_ROOT / paths['data_dir']).resolve()
HMM_DIR = (PROJECT_ROOT / paths['hmm_models_dir']).resolve()
RESULTS_DIR = (PROJECT_ROOT / paths['results_dir']).resolve()

K = int(analysis['K'])
method = analysis['method']
n_timepoints = int(analysis['n_timepoints'])
use_gamma = bool(analysis['use_gamma'])
min_eff_n = int(analysis.get('min_eff_n', 5))
min_hard_n = int(analysis.get('min_hard_n', 5))

with open(RESULTS_DIR / 'analysis_metadata.json', encoding='utf-8') as f:
    meta = json.load(f)

use_example_data = bool(meta.get('use_example_data', use_example_data))
subjects = meta['subjects']
select_networks = np.array(meta['select_networks'])

roi_dir = RESULTS_DIR / '01_roi'
sort_idx = np.load(roi_dir / 'sort_idx.npy')

if use_example_data:
    Y = pd.read_csv(DATA_DIR / 'parcel_timeseries.csv').values[:, sort_idx]
    decode_npz = np.load(DATA_DIR / 'hmm_decode.npz')
    Gamma = decode_npz['Gamma']
    vpath = decode_npz['vpath']
    hmm = None
    print('Example mode: loaded precomputed HMM decode arrays.')
else:
    from glhmm import statistics

    signal_file = DATA_DIR / f'all_schaefer_parcel1000_{method}.csv'
    Y = pd.read_csv(signal_file).values[:, sort_idx]
    with open(HMM_DIR / f'trained_hmm_1000parcels_{K}states_{method}.pkl', 'rb') as f:
        hmm = pickle.load(f)
    indices = statistics.get_indices_timestamp(n_timepoints, len(subjects))


Example mode: loaded precomputed HMM decode arrays.


In [2]:
if not use_example_data:
    Gamma, Xi, _ = hmm.decode(X=None, Y=Y, indices=indices, viterbi=False)
    vpath = hmm.decode(X=None, Y=Y, indices=indices, viterbi=True)

Y_by_subj = {}
gamma_by_subj = {}
states_by_subj = {}
start = 0
for subj in subjects:
    Y_by_subj[subj] = Y[start : start + n_timepoints, :]
    gamma_by_subj[subj] = Gamma[start : start + n_timepoints, :]
    states_by_subj[subj] = np.argmax(vpath[start : start + n_timepoints, :], axis=1)
    start += n_timepoints

print(f'Prepared {len(subjects)} subjects.')


Prepared 3 subjects.


In [3]:
def compute_weighted_correlation(timeseries: np.ndarray, weights: np.ndarray) -> np.ndarray:
    w = np.clip(weights.astype(float), 0.0, np.inf)
    if np.all(w == 0):
        return np.full((timeseries.shape[1], timeseries.shape[1]), np.nan)
    w = w / (w.sum() + 1e-12)
    mean_w = (timeseries * w[:, None]).sum(axis=0)
    Xc = timeseries - mean_w
    cov_w = (Xc * w[:, None]).T @ Xc
    var = np.diag(cov_w)
    denom = np.sqrt(np.outer(var, var)) + 1e-12
    return np.clip(cov_w / denom, -1.0, 1.0)


def effective_sample_size(weights: np.ndarray) -> float:
    w = weights.astype(float)
    if np.all(w == 0):
        return 0.0
    return (w.sum() ** 2) / ((w ** 2).sum() + 1e-12)


per_subject_state_FC: Dict[str, Dict[int, np.ndarray]] = {}
summary_rows = []

for subj, Y_subj in Y_by_subj.items():
    T_i, n_regions = Y_subj.shape
    per_subject_state_FC[subj] = {}
    if use_gamma:
        gamma = gamma_by_subj[subj]
        for k in range(K):
            w = gamma[:, k]
            neff = effective_sample_size(w)
            FC = (
                np.full((n_regions, n_regions), np.nan)
                if neff < min_eff_n
                else compute_weighted_correlation(Y_subj, w)
            )
            per_subject_state_FC[subj][k] = FC
            summary_rows.append({'subj': subj, 'state': k + 1, 'method': 'soft', 'eff_n_or_n': float(neff)})
    else:
        z = states_by_subj[subj]
        for k in range(K):
            idx = z == k
            n_used = int(idx.sum())
            FC = (
                np.full((n_regions, n_regions), np.nan)
                if n_used < min_hard_n
                else np.corrcoef(Y_subj[idx], rowvar=False)
            )
            per_subject_state_FC[subj][k] = FC
            summary_rows.append({'subj': subj, 'state': k + 1, 'method': 'hard', 'eff_n_or_n': float(n_used)})

summary_df = pd.DataFrame(summary_rows)
print(summary_df.head())

out_dir = RESULTS_DIR / '02_state_fc'
out_dir.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    out_dir / 'per_subject_state_FC.npz',
    allow_pickle=True,
    per_subject_state_FC=per_subject_state_FC,
    region_names=select_networks,
)
summary_df.to_csv(out_dir / 'per_subject_state_FC_summary.csv', index=False)
print(f'Saved state FC → {out_dir}')


     subj  state method  eff_n_or_n
0  sub-01      1   soft   84.961642
1  sub-01      2   soft   74.537366
2  sub-01      3   soft   70.522867
3  sub-02      1   soft   87.467537
4  sub-02      2   soft   79.356552
Saved state FC → D:\文件同步\文件\数据分析结果\data_code_figure\adaptive_reserve_moderate\graph_theory\results\example\02_state_fc
